# Efficient VLM Inference via Visual Token Compression

Colab demo for Qwen2.5-VL / Qwen2-VL inference-time visual token compression benchmarks.

## 1. Install dependencies

Use an A100 runtime when possible. Colab normally ships with a CUDA-matched PyTorch build, so the torch install line is kept as an optional fallback.

In [ ]:
try:
    import torch
except ImportError:
    %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124

%pip install -q -U torchvision transformers accelerate qwen-vl-utils datasets pillow pandas matplotlib tqdm psutil pynvml pyyaml

# If you see KeyError: 'qwen2_5_vl' or 'qwen2_vl', run this and restart runtime:
# %pip install -q -U git+https://github.com/huggingface/transformers accelerate

## 2. Move to project root

If you uploaded this folder to `/content/vlm_token_compression`, this cell will enter it. Otherwise, edit `PROJECT_DIR`.

In [ ]:
import os

PROJECT_DIR = "/content/vlm_token_compression"
if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
print("Current directory:", os.getcwd())

## 3. Check GPU

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("Memory GB:", round(props.total_memory / 1024**3, 1))
!nvidia-smi

## 4. Smoke test compression functions without loading the VLM

In [ ]:
import torch
from src.compression import create_compression_method

device = "cuda" if torch.cuda.is_available() else "cpu"
tokens = torch.randn(1, 64, 128, device=device)
for name in ["none", "fixed", "importance", "merging"]:
    method = create_compression_method(name, retention_ratio=0.5, apply_proxy_image_budget=False)
    compressed = method.compress_visual_tokens(tokens).tokens
    print(name, tuple(tokens.shape), "->", tuple(compressed.shape))

## 5. Quick benchmark

This loads the model and runs a tiny benchmark on synthetic images. On A100, start with Qwen2.5-VL-3B. If model download or memory is a problem, use the Qwen2-VL-2B fallback cell.

In [ ]:
!python run_benchmark.py --quick --model-id Qwen/Qwen2.5-VL-3B-Instruct --dtype bf16

In [ ]:
# Fallback option:
# !python run_benchmark.py --quick --model-id Qwen/Qwen2-VL-2B-Instruct --dtype fp16

## 6. Full benchmark

In [ ]:
# This can take a while. Reduce --samples or remove high/4-image settings if needed.
# !python run_benchmark.py \
#   --methods none,fixed,importance,merging \
#   --ratios 1.0,0.75,0.5,0.25,0.1 \
#   --resolutions low,medium,high \
#   --num-images 1,2,4 \
#   --samples 3 \
#   --max-new-tokens 64

## 7. View results and plots

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

raw_path = "results/benchmark_results.csv"
summary_path = "results/summary_results.csv"
display(pd.read_csv(raw_path).head())
display(pd.read_csv(summary_path))

!python plot_results.py
for filename in [
    "latency_vs_retention_ratio.png",
    "memory_vs_retention_ratio.png",
    "quality_vs_retention_ratio.png",
    "efficiency_accuracy_tradeoff.png",
]:
    display(IPImage(filename=f"results/{filename}"))